# 📊 Análisis de Inflación en Argentina
**Autor:** Marcos Javier Ibañez  
**Herramientas:** Python · Pandas · Matplotlib · Requests  

Este notebook analiza la evolución del IPC (Índice de Precios al Consumidor) en Argentina usando datos oficiales del INDEC.
Se incluye:
- Descarga automática de datos desde la API del BCRA
- Evolución mensual y acumulada de la inflación
- Promedio móvil para suavizar la serie
- Detección de picos inflacionarios
- Comparación por períodos

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficos
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

print('✅ Librerías cargadas correctamente')

## 2. Descarga de datos desde la API del BCRA

Usamos la API pública del Banco Central de la República Argentina (BCRA).  
La variable `7` corresponde a la **inflación mensual (IPC - variación % mensual)**.

In [ ]:
def obtener_datos_bcra(id_variable, desde='2019-01-01', hasta='2025-12-31'):
    """
    Descarga datos de la API pública del BCRA.
    Documentación: https://api.bcra.gob.ar
    """
    url = f'https://api.bcra.gob.ar/estadisticas/v2.0/DatosVariable/{id_variable}/{desde}/{hasta}'
    headers = {'accept': 'application/json'}
    
    try:
        response = requests.get(url, headers=headers, timeout=10, verify=False)
        data = response.json()
        df = pd.DataFrame(data['results'])
        df['fecha'] = pd.to_datetime(df['fecha'])
        df = df.sort_values('fecha').reset_index(drop=True)
        df = df.rename(columns={'valor': 'inflacion_mensual'})
        print(f'✅ Datos descargados: {len(df)} registros ({df.fecha.min().strftime("%b %Y")} → {df.fecha.max().strftime("%b %Y")})')
        return df
    except Exception as e:
        print(f'⚠️  No se pudo conectar a la API. Usando datos de ejemplo. Error: {e}')
        return None

df = obtener_datos_bcra(id_variable=27)  # Variable 27 = IPC Nivel General mensual

# Fallback: datos manuales si la API no responde
if df is None:
    datos_manuales = {
        'fecha': pd.date_range(start='2020-01-01', periods=60, freq='MS'),
        'inflacion_mensual': [
            2.3,2.0,3.3,1.5,1.5,2.2,1.9,2.7,2.8,3.8,3.2,4.0,
            4.0,3.6,4.8,4.1,3.3,3.2,3.0,2.5,3.5,3.5,2.4,3.8,
            4.7,4.9,6.7,6.0,5.1,5.3,7.4,7.0,6.2,6.3,4.9,5.1,
            6.0,6.6,7.7,8.4,8.3,7.8,6.3,7.0,12.4,6.2,4.9,8.3,
            6.0,13.2,8.4,7.3,12.8,20.6,25.5,10.7,8.3,4.2,2.4,3.7
        ]
    }
    df = pd.DataFrame(datos_manuales)
    print('✅ Datos de ejemplo cargados')

df.head()

## 3. Procesamiento de datos

In [ ]:
# Inflación acumulada anual (producto de variaciones mensuales)
df['año'] = df['fecha'].dt.year

inflacion_anual = (
    df.groupby('año')['inflacion_mensual']
    .apply(lambda x: (np.prod(1 + x/100) - 1) * 100)
    .reset_index()
    .rename(columns={'inflacion_mensual': 'inflacion_anual'})
)

# Promedio móvil de 3 meses (suaviza ruido)
df['media_movil_3m'] = df['inflacion_mensual'].rolling(window=3).mean()

# Máximo histórico del período
idx_max = df['inflacion_mensual'].idxmax()
fecha_max = df.loc[idx_max, 'fecha'].strftime('%B %Y')
valor_max = df.loc[idx_max, 'inflacion_mensual']

print(f'📌 Pico inflacionario del período: {valor_max:.1f}% en {fecha_max}')
print(f'📌 Promedio mensual del período: {df["inflacion_mensual"].mean():.1f}%')
print()
print('Inflación acumulada por año:')
print(inflacion_anual.to_string(index=False))

## 4. Visualización: Evolución mensual del IPC

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))

ax.bar(df['fecha'], df['inflacion_mensual'], color='steelblue', alpha=0.6, width=20, label='Inflación mensual')
ax.plot(df['fecha'], df['media_movil_3m'], color='crimson', linewidth=2.5, label='Promedio móvil 3 meses')

# Marcar el pico
ax.annotate(
    f'Pico: {valor_max:.1f}%',
    xy=(df.loc[idx_max, 'fecha'], valor_max),
    xytext=(df.loc[idx_max, 'fecha'], valor_max + 2),
    fontsize=10, color='darkred', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='darkred')
)

ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.set_title('Inflación Mensual en Argentina (IPC - Nivel General)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Fecha')
ax.set_ylabel('Variación mensual (%)')
ax.legend()
plt.tight_layout()
plt.savefig('inflacion_mensual.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado como inflacion_mensual.png')

## 5. Visualización: Inflación acumulada por año

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colores = ['#d32f2f' if v > 100 else '#f57c00' if v > 50 else '#388e3c' for v in inflacion_anual['inflacion_anual']]
bars = ax.bar(inflacion_anual['año'].astype(str), inflacion_anual['inflacion_anual'], color=colores, edgecolor='white', linewidth=0.5)

# Etiquetas encima de cada barra
for bar, val in zip(bars, inflacion_anual['inflacion_anual']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val:.0f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('Inflación Acumulada Anual en Argentina', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Año')
ax.set_ylabel('Inflación anual (%)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

# Leyenda de colores
from matplotlib.patches import Patch
leyenda = [Patch(color='#d32f2f', label='>100%'), Patch(color='#f57c00', label='50–100%'), Patch(color='#388e3c', label='<50%')]
ax.legend(handles=leyenda, title='Nivel')

plt.tight_layout()
plt.savefig('inflacion_anual.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado como inflacion_anual.png')

## 6. Tabla resumen

In [ ]:
resumen = df.groupby('año')['inflacion_mensual'].agg(
    Promedio_mensual='mean',
    Maximo='max',
    Minimo='min',
    Desvio_std='std'
).round(2)

resumen = resumen.merge(inflacion_anual.set_index('año'), left_index=True, right_index=True)
resumen = resumen.rename(columns={'inflacion_anual': 'Acumulada_anual_%'})
resumen['Acumulada_anual_%'] = resumen['Acumulada_anual_%'].round(1)

print('📋 Resumen estadístico por año:')
print(resumen.to_string())

## 7. Conclusiones

- La inflación argentina muestra una **tendencia estructuralmente creciente** desde 2020, con aceleración marcada a partir de 2022.
- El pico mensual del período se registró en **diciembre 2023 (25.5%)**, asociado al shock de precios post-devaluación.
- A partir de 2024, se observa una **desaceleración gradual**, aunque los niveles siguen siendo elevados en perspectiva histórica.
- El análisis de **series de tiempo** (promedio móvil) permite filtrar la volatilidad mensual y visualizar la tendencia subyacente.

---
*Datos: API pública BCRA — [https://api.bcra.gob.ar](https://api.bcra.gob.ar)*  
*Autor: Marcos Javier Ibañez — [LinkedIn](https://www.linkedin.com/in/ibañez-marcos-cv/)*